# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [1]:
# %pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

from transformers import TrainingArguments as HFTrainingArguments, Trainer, AutoModelForSequenceClassification
from tqdm.auto import tqdm
import os

### Data Preparation

In [6]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [7]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: gchhablani/bert-base-cased-finetuned-qqp
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [9]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [10]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [11]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [12]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [9]:
bert_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=64,
    shuffle=False,
    collate_fn=transformers.default_data_collator,
    num_workers=2,
)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
model = model.to(device)

_ = model.eval()

In [10]:
correct = 0
total = 0
with torch.no_grad():
    for batch in tqdm(bert_loader, desc="val"):
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch["token_type_ids"],
        )
        pred = torch.argmax(out.logits, dim=1)
        correct += (pred == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

accuracy = correct / total

val:   0%|          | 0/632 [00:00<?, ?it/s]

In [11]:
print(accuracy)
assert 0.9 < accuracy < 0.91

0.9083848627256987


### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [19]:
model_name = "microsoft/deberta-v3-base"
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   100%|          | 198/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias          

In [13]:
print(os.environ.get("CUDA_HOME"))

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    correct = (preds == labels).sum()
    total = len(labels)
    return {"accuracy": correct / total}

training_args = HFTrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    dataloader_num_workers=2,
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=qqp_preprocessed["train"],
    eval_dataset=qqp_preprocessed["validation"],
    compute_metrics=compute_metrics
)

trainer.train()
eval_results = trainer.evaluate()
eval_results

/usr/local/cuda


Loading weights:   100%|          | 198/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias          

Step,Training Loss
500,0.656544
1000,0.000000
1500,0.000000
2000,0.000000
2500,0.000000
3000,0.000000
3500,0.000000
4000,0.000000
4500,0.000000
5000,0.000000


Writing model shards:   100%|          | 1/1 [00:00<?, ?it/s]

...

Writing model shards:   100%|          | 1/1 [00:00<?, ?it/s]

Writing model shards:   100%|          | 1/1 [00:00<?, ?it/s]

Writing model shards:   100%|          | 1/1 [00:00<?, ?it/s]

Writing model shards:   100%|          | 1/1 [00:00<?, ?it/s]

Writing model shards:   100%|          | 1/1 [00:00<?, ?it/s]

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [22]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
tknzer = transformers.AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = model.to(device)

train_texts = list(
    set(
        [ex["text1"] for ex in qqp_preprocessed["train"]]
        + [ex["text2"] for ex in qqp_preprocessed["train"]]
    )
)

batch_size = 16
all_embeddings = []
model.eval()


# works for Bert/DeBERTa/etc
encoder = getattr(model, model.base_model_prefix)

for i in tqdm(range(0, len(train_texts), batch_size), desc="embed"):
    batch_texts = train_texts[i : i + batch_size]
    inputs = tknzer(batch_texts, padding=True, truncation=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        # outputs = model.deberta(**inputs)
        # batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu()
        enc_out = encoder(**inputs)
        batch_embeddings = enc_out.last_hidden_state[:, 0, :].cpu()
        all_embeddings.append(batch_embeddings)
train_embeddings = torch.cat(all_embeddings, dim=0)

embed:   100%|          | 30868/30868 [00:00<?, ?it/s]

In [23]:
def find_dublicates(questions):
    inp = tknzer(questions, padding=True, truncation=True, return_tensors="pt")
    inp = {k: v.to(device) for k, v in inp.items()}
    model.eval()
    with torch.no_grad():
        enc = model.deberta(**inp)
        embeddings = enc.last_hidden_state[:, 0, :].cpu()
    cos_sim = F.cosine_similarity(
        embeddings.unsqueeze(1),
        train_embeddings.unsqueeze(0),
        dim=2,
    )
    return cos_sim.topk(5, dim=1)


def print_duplicates(questions):
    _, indices = find_dublicates(questions)
    for i, question in enumerate(questions):
        print(f"Дубликаты из тренировочного набора для вопроса \"{question}\":")
        for j, idx in enumerate(indices[i]):
            print(f"{j + 1}. {train_texts[idx.item()]}")
        print()


print_duplicates(
    [
        "How do I learn Python?",
        "What is the best way to lose weight?",
        "How can I improve my English speaking skills?",
        "What are some tips for a successful job interview?",
        "How do I make money online?",
    ]
)

Дубликаты из тренировочного набора для вопроса "How do I learn Python?":
1. How do I increase my discipline?
2. Why do people pray?
3. How do I start web development?
4. How do I make brownies?
5. How do I calculate IQ?

Дубликаты из тренировочного набора для вопроса "What is the best way to lose weight?":
1. What is the best way to get a scholarship?
2. How is marriage defined in Islam?
3. How do I stay motivated to attain my goals?
4. What does it cost to develop an app like WhatsApp?
5. Have you ever cheated on your partner? If yes, then how often and how?

Дубликаты из тренировочного набора для вопроса "How can I improve my English speaking skills?":
1. What is it like being a police officer?
2. How can I improve my drawing skills?
3. How can I control my chatter girlfriend?
4. How do I improve my thought quality?
5. How can I improve my painting skills?

Дубликаты из тренировочного набора для вопроса "What are some tips for a successful job interview?":
1. What are some good intro

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part